<a href="https://colab.research.google.com/github/HYUNSUNG03/Ai_Programming/blob/main/MakeData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faker openpyxl requests -q

In [ ]:
import re
import random
import requests
import pandas as pd
from faker import Faker

fake = Faker('ko_KR')
print("✅ 라이브러리 로드 완료!")

In [ ]:
#name
def build_names_dataset(n=1000):
    names = list(set([fake.name() for _ in range(n)]))
    df = pd.DataFrame({'이름': names})
    print(f"✅ 이름 {len(df)}개 생성")
    return df

df_names = build_names_dataset(1000)
print(df_names.head(5))

In [ ]:
#PhoneNumber
def build_phones_dataset(n=500):
    phones = set()

    while len(phones) < n:
        sep = random.choice(['-', ' ', ''])

        phone_type = random.choice(['휴대폰', '지역번호'])

        if phone_type == '휴대폰':
            # 010만 (011, 016 등은 현재 거의 사용 안 함)
            mid = str(random.randint(1000, 9999))
            end = str(random.randint(1000, 9999))
            phones.add(f"010{sep}{mid}{sep}{end}")

        elif phone_type == '지역번호':
            area = random.choice([
                '02',   # 서울
                '031',  # 경기
                '032',  # 인천
                '033',  # 강원
                '041',  # 충남
                '042',  # 대전
                '043',  # 충북
                '044',  # 세종
                '051',  # 부산
                '052',  # 울산
                '053',  # 대구
                '054',  # 경북
                '055',  # 경남
                '061',  # 전남
                '062',  # 광주
                '063',  # 전북
                '064',  # 제주
            ])

            if area == '02':
                # 서울은 국번 3자리
                mid = str(random.randint(100, 999))
            else:
                # 그 외는 국번 3~4자리
                mid = str(random.randint(100, 9999))

            end = str(random.randint(1000, 9999))
            phones.add(f"{area}{sep}{mid}{sep}{end}")

    df = pd.DataFrame({'전화번호': list(phones)[:n]})
    print(f"✅ 전화번호 {len(df)}개 생성")
    return df

df_phones = build_phones_dataset(500)
print(df_phones.head(10))

In [ ]:
#PersonalNumber
def build_rrn_dataset(n=300):
    rrns = set()
    while len(rrns) < n:
        if random.random() < 0.5:
            year   = str(random.randint(0, 99)).zfill(2)
            gender = random.choice(['1', '2'])
        else:
            year   = str(random.randint(0, 25)).zfill(2)
            gender = random.choice(['3', '4'])

        month = str(random.randint(1, 12)).zfill(2)
        day   = str(random.randint(1, 28)).zfill(2)
        tail  = str(random.randint(100000, 999999))

        for sep in ['-', '']:
            rrns.add(f"{year}{month}{day}{sep}{gender}{tail}")

    df = pd.DataFrame({'주민등록번호': list(rrns)[:n]})
    print(f"✅ 주민등록번호 {len(df)}개 생성")
    return df

df_rrns = build_rrn_dataset(300)
print(df_rrns.head(5))

In [ ]:
#Account
def build_accounts_dataset(n=500):
    accounts = set()
    # 실제 은행별 계좌번호 형식 반영
    formats = [
        lambda: f"{random.randint(100,999)}-{random.randint(10,99)}-{random.randint(100000,999999)}",       # 국민
        lambda: f"{random.randint(100,999)}-{random.randint(100,999)}-{random.randint(100000,999999)}",     # 신한
        lambda: f"{random.randint(1000,9999)}-{random.randint(1000,9999)}-{random.randint(1000,9999)}",    # 우리
        lambda: f"{random.randint(100,999)}-{random.randint(10,99)}-{random.randint(100000,999999)}-{random.randint(10,99)}", # 하나
        lambda: f"{random.randint(10,99)}-{random.randint(1000,9999)}-{random.randint(1000,9999)}-{random.randint(10,99)}",  # 카카오
    ]
    while len(accounts) < n:
        accounts.add(random.choice(formats)())

    df = pd.DataFrame({'계좌번호': list(accounts)[:n]})
    print(f"✅ 계좌번호 {len(df)}개 생성")
    return df

df_accounts = build_accounts_dataset(500)
print(df_accounts.head(5))

In [ ]:
#email
def build_emails_dataset(n=500):
    domains = ['naver.com', 'gmail.com', 'kakao.com',
               'daum.net', 'hanmail.net', 'nate.com',
               'hotmail.com', 'outlook.com']
    emails = set()
    while len(emails) < n:
        user = fake.user_name().replace('-', '.')
        emails.add(f"{user}@{random.choice(domains)}")

    df = pd.DataFrame({'이메일': list(emails)[:n]})
    print(f"✅ 이메일 {len(df)}개 생성")
    return df

df_emails = build_emails_dataset(500)
print(df_emails.head(5))

In [ ]:
#Address
JUSO_API_KEY = "여기에_승인키_입력"

def fetch_addresses(keyword, count=20):
    url = "https://business.juso.go.kr/addrlink/addrLinkApi.do"
    params = {
        "confmKey":     JUSO_API_KEY,
        "currentPage":  1,
        "countPerPage": count,
        "keyword":      keyword,
        "resultType":   "json"
    }
    try:
        res  = requests.get(url, params=params, timeout=5)
        data = res.json()
        juso = data["results"]["juso"]
        return [j["roadAddr"] for j in juso if j["roadAddr"]]
    except Exception as e:
        print(f"  오류 ({keyword}): {e}")
        return []


def build_address_dataset():
    keywords = [
        "강남", "서초", "마포", "송파", "종로",
        "해운대", "수성", "판교", "분당", "수원",
        "인천", "대전", "광주", "부산", "대구",
        "일산", "평택", "천안", "전주", "제주"
    ]
    pool = []
    for kw in keywords:
        addrs = fetch_addresses(kw, count=20)
        pool.extend(addrs)
        print(f"  '{kw}' → {len(addrs)}개 수집")

    pool = list(set(pool))
    df   = pd.DataFrame({'주소': pool})
    print(f"\n✅ 주소 총 {len(df)}개 수집 완료!")
    return df

df_addresses = build_address_dataset()
print(df_addresses.head(5))

In [ ]:
#Ip
def build_ip_dataset(n=300):
    ips = set()
    while len(ips) < n:
        ips.add(
            f"{random.randint(1,254)}."
            f"{random.randint(0,255)}."
            f"{random.randint(0,255)}."
            f"{random.randint(1,254)}"
        )
    df = pd.DataFrame({'IP주소': list(ips)[:n]})
    print(f"✅ IP주소 {len(df)}개 생성")
    return df

df_ips = build_ip_dataset(300)
print(df_ips.head(5))

In [ ]:
#DownloadExcel
from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/privacy_dataset.xlsx"

with pd.ExcelWriter(save_path, engine='openpyxl') as writer:
    df_names   .to_excel(writer, sheet_name='이름',       index=False)
    df_phones  .to_excel(writer, sheet_name='전화번호',   index=False)
    df_rrns    .to_excel(writer, sheet_name='주민등록번호', index=False)
    df_accounts.to_excel(writer, sheet_name='계좌번호',   index=False)
    df_emails  .to_excel(writer, sheet_name='이메일',     index=False)
    df_addresses.to_excel(writer, sheet_name='주소',      index=False)
    df_ips     .to_excel(writer, sheet_name='IP주소',     index=False)

print(f"✅ 엑셀 저장 완료: {save_path}")
print(f"\n📊 시트별 데이터 수:")
print(f"   이름:        {len(df_names)}개")
print(f"   전화번호:    {len(df_phones)}개")
print(f"   주민등록번호: {len(df_rrns)}개")
print(f"   계좌번호:    {len(df_accounts)}개")
print(f"   이메일:      {len(df_emails)}개")
print(f"   주소:        {len(df_addresses)}개")
print(f"   IP주소:      {len(df_ips)}개")